# 🥇 DataCo Supply Chain Gold Layer & ML Inference Pipeline
---

### 📌 Architecture Overview
This notebook implements the **Gold Layer Modeling & Machine Learning Inference** phase. It ingests clean transactional logs from the Silver Delta table, reconstructs the feature engineering pipeline, loads your serialized **LightGBM ML model** to predict delivery risks, splits the enriched dataset into an optimized **Star Schema**, and writes them as Gold Delta tables.

* **Source Table:** `workspace.dataco_silver.supply_chain_cleaned` (Delta Lake)
* **Target Schema:** `workspace.dataco_gold` (Fact & Dimensions)
* **ML Model:** AutoGluon LightGBM (`LightGBM_r163_BAG_L1` ~5MB CV folds)

```
 +-----------------------------+      +---------------------------+      +-------------------------------+
 |  Silver Layer (Delta Lake)  |      |    Databricks Workspace   |      |    Gold Layer (Delta Lake)    |
 |                             |      |                           |      |                               |
 |    supply_chain_cleaned     | ---> |  Replicate Feature Prep   | ---> |  fact_sales (ML enriched)     |
 |   (clean normalized data)   |      |  & LightGBM Prediction    |      |  dim_customers, dim_products  |
 +-----------------------------+      +---------------------------+      +-------------------------------+
```

---
### ⚙️ Implementation Steps
1. **Configuration Widgets:** Defines parameter widgets for schemas, catalog, and Unity Catalog volume paths.
2. **Load Silver Table:** Reads the cleaned Silver delta table into a Spark DataFrame.
3. **Feature Engineering Parity:** Reads the `preprocessing_maps.json` metadata file to apply identical target encodings and rare label whitelists (`__OTHER__`) as local training.
4. **LightGBM Batch Inference:** Installs the required libraries and runs batch prediction to generate actual predicted risks and probability scores.
5. **Star Schema Transformation:** Splits the data into a central `fact_sales` and three dimension tables: `dim_customers`, `dim_products`, and `dim_date`.
6. **Gold Delta Storage:** Saves tables in Delta format and runs compactions and Z-ordering indexes.
7. **Validation & Quality Assertions:** Automatically checks row counts, null constraints, and key integrity.


## 🛠️ Step 1: Configuration & Parameterization
Define widgets to pass parameters like schema catalog names and Unity Catalog Volume paths dynamically.


In [0]:
# Initialize Spark context first (required on serverless)
spark

# Define widgets with defaults matching your Databricks cluster and UC Volume storage
dbutils.widgets.text("source_catalog", "workspace", "Source Catalog (Silver)")
dbutils.widgets.text("source_schema", "dataco_silver", "Source Schema (Silver)")
dbutils.widgets.text("source_table", "supply_chain_cleaned", "Source Table Name (Silver)")

dbutils.widgets.text("target_catalog", "workspace", "Target Catalog (Gold)")
dbutils.widgets.text("target_schema", "dataco_gold", "Target Schema (Gold)")
dbutils.widgets.dropdown("write_mode", "overwrite", ["overwrite", "append"], "Write Mode")

# Paths inside Databricks UC Volumes where model folds and mapping JSON are stored
dbutils.widgets.text("model_volume_path", "/Volumes/workspace/dataco_silver/models/", "UC Volume Model Folder")
dbutils.widgets.text("maps_volume_path", "/Volumes/workspace/dataco_silver/models/preprocessing_maps.json", "UC Volume Preprocess JSON Map")


## 📖 Step 2: Read Silver Table
Load parameter values and query the Silver Delta table into a Spark DataFrame.


In [0]:
# Ensure Spark is initialized (required on serverless)
spark.sql("SELECT 1").collect()

# Retrieve parameter values from widgets
source_catalog = dbutils.widgets.get("source_catalog").strip()
source_schema = dbutils.widgets.get("source_schema").strip()
source_table = dbutils.widgets.get("source_table").strip()

target_catalog = dbutils.widgets.get("target_catalog").strip()
target_schema = dbutils.widgets.get("target_schema").strip()
write_mode = dbutils.widgets.get("write_mode").strip()

model_volume_path = dbutils.widgets.get("model_volume_path").strip()
maps_volume_path = dbutils.widgets.get("maps_volume_path").strip()

# Build full source path and load the Silver table
if source_catalog.lower() != "hive_metastore":
    source_path = f"`{source_catalog}`.`{source_schema}`.`{source_table}`"
else:
    source_path = f"`{source_schema}`.`{source_table}`"

print(f"📖 Reading Silver Delta table: {source_path}")
df_silver = spark.read.table(source_path)
print(f"✅ Loaded {df_silver.count():,} records from Silver.")


📖 Reading Silver Delta table: `workspace`.`dataco_silver`.`supply_chain_cleaned`
✅ Loaded 180,519 records from Silver.


## ✏️ Step 3: Feature Engineering with Parity Maps
Loads the serialized `preprocessing_maps.json` metadata from the volume. Re-extracts temporal fields, financial ratios, rare-label collapsing whitelists (`__OTHER__`), and smoothed target encodings to ensure 100% mathematical parity with your local model training splits.


In [0]:
import json
import pandas as pd

print(f"📂 Loading preprocessing metadata map from: {maps_volume_path}")
with open(maps_volume_path, 'r', encoding='utf-8') as f:
    maps_data = json.load(f)

global_mean = maps_data["global_mean"]
print(f"✅ Target global mean: {global_mean:.6f}")

# Convert Spark DataFrame to Pandas for exact, unified pandas-level feature engineering
print("🔄 Converting Spark DataFrame to Pandas for feature engineering...")
df_pandas = df_silver.toPandas()

# 1. Rename column to bypass date-coercion regex bug
df_pandas = df_pandas.rename(columns={"days_for_shipment_scheduled": "scheduled_shipping_duration"})

# 2. Extract Temporal Features
print("⏱️ Extracting temporal fields (month, day of week, hour)...")
dates = pd.to_datetime(df_pandas['order_date'])
df_pandas['order_month'] = dates.dt.month.astype('int8')
df_pandas['order_day_of_week'] = dates.dt.dayofweek.astype('int8')
df_pandas['order_hour'] = dates.dt.hour.astype('int8')

# 3. Extract Financial Prioritization Ratios
print("💰 Calculating financial ratios...")
df_pandas['discount_ratio'] = (df_pandas['order_item_discount'] / (df_pandas['sales'] + 1)).astype('float32')
df_pandas['profit_margin'] = (df_pandas['benefit_per_order'] / (df_pandas['sales'] + 1)).astype('float32')

# 4. Apply Rare Label Whitelists (Collapsing other low frequency categories to __OTHER__)
print("🗂️ Applying rare label whitelists...")
rare_cols = ['customer_fname', 'customer_lname', 'order_country']
for col in rare_cols:
    whitelist = set(maps_data['rare_label_whitelists'][col])
    df_pandas[col] = df_pandas[col].astype(str).apply(lambda x: x if x in whitelist else '__OTHER__')

# 5. Apply Target Encoding Mappings
print("🧬 Applying smoothed target encoding maps...")
target_cols = ['order_city', 'customer_city', 'product_name', 'category_name']
for col in target_cols:
    mapping = maps_data['encoding_maps'][col]
    df_pandas[f'{col}_encoded'] = df_pandas[col].astype(str).map(mapping).fillna(global_mean).astype('float32')

print(f"✅ Feature engineering completed. Shape: {df_pandas.shape}")


📂 Loading preprocessing metadata map from: /Volumes/workspace/dataco_silver/models/preprocessing_maps.json
✅ Target global mean: 0.549090
🔄 Converting Spark DataFrame to Pandas for feature engineering...
⏱️ Extracting temporal fields (month, day of week, hour)...
💰 Calculating financial ratios...
🗂️ Applying rare label whitelists...
🧬 Applying smoothed target encoding maps...
✅ Feature engineering completed. Shape: (180519, 60)


## 🤖 Step 4: AutoGluon LightGBM Batch Inference
Installs `autogluon.tabular`, checks for library verification, loads the 5MB LightGBM folds model, and runs batch prediction on the pandas features to output predictions and probability scores.


In [0]:
%python
%pip install autogluon.tabular==1.5.0 numpy>=2.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 MB 104.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 21.0.0
    Not uninstalling pyarrow at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-1ce0c95f-2506-4f34-a0ad-1aa885790b3e
    Can't uninstall 'pyarrow'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%python
%pip install lightgbm

import sys
from autogluon.tabular import TabularPredictor

# 1. Environment & Pickle check
print(f"🐍 Python version of Databricks: {sys.version}")
print(f"📂 Loading TabularPredictor from: {model_volume_path}")
predictor = TabularPredictor.load(model_volume_path, require_py_version_match=False)

# 2. Run batch prediction using ONLY the LightGBM booster model
print("🚀 Running batch prediction on LightGBM folds...")
model_names = predictor.model_names()
gbm_models = [m for m in model_names if "LightGBM" in m]
if len(gbm_models) > 0:
    model_name = gbm_models[0]
    print(f"Automatically resolved LightGBM model name: {model_name}")
else:
    model_name = predictor.model_best
    print(f"LightGBM model not found in model list. Falling back to best model: {model_name}")
predictions = predictor.predict(df_pandas, model=model_name)
probabilities = predictor.predict_proba(df_pandas, model=model_name)

# 3. Append predictions and target class 1 (Late Delivery) probabilities
df_pandas['predicted_late_delivery_risk'] = predictions.astype('int8')
df_pandas['predicted_late_delivery_probability'] = probabilities[1].astype('float32')

print("✅ Inference completed successfully!")
print(df_pandas[['order_id', 'late_delivery_risk', 'predicted_late_delivery_risk', 'predicted_late_delivery_probability']].head(5))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 69.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
🐍 Python version of Databricks: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
📂 Loading TabularPredictor from: /Volumes/workspace/dataco_silver/models/models


Found 2 mismatches between original and current metadata:
	INFO: AutoGluon Python micro version mismatch (original=3.12.13, current=3.12.3)


🚀 Running batch prediction on LightGBM folds...
✅ Inference completed successfully!
   order_id  ...  predicted_late_delivery_probability
0     77202  ...                             0.493619
1     75923  ...                             0.762494
2     75902  ...                             0.636042
3      5895  ...                             0.623706
4     55984  ...                             0.464247

[5 rows x 4 columns]


## 🔀 Step 5: Star Schema Split
Converts the Pandas DataFrame back to PySpark and splits the enriched dataset into four standardized relational tables: `dim_customers`, `dim_products`, `dim_date`, and `fact_sales`.


In [0]:
from pyspark.sql.functions import col, concat_ws, to_date, year, month, quarter, dayofweek, date_format, lit, when

# Extract predictions and join them back to the original Spark df_silver to preserve original names and metrics
print("🔄 Extracting predictions and merging back to the original Silver Spark DataFrame...")
df_pred_spark = spark.createDataFrame(df_pandas[['order_item_id', 'predicted_late_delivery_risk', 'predicted_late_delivery_probability']])
df_gold = df_silver.join(df_pred_spark, "order_item_id", "inner")

# 1. CREATE DIM_CUSTOMERS (Unique Customer Details using original uncollapsed fields)
print("👥 Constructing dim_customers table...")
dim_customers = (df_gold
    .select(
        col("customer_id").cast("int").alias("customer_id"),
        concat_ws(" ", col("customer_fname"), col("customer_lname")).alias("customer_name"),
        col("customer_segment"),
        col("customer_city"),
        col("customer_state"),
        col("customer_country"),
        col("customer_zipcode"),
        col("latitude").cast("double").alias("latitude"),
        col("longitude").cast("double").alias("longitude")
    )
    .dropDuplicates(["customer_id"])
)

# 2. CREATE DIM_PRODUCTS (Unique Product Details)
print("📦 Constructing dim_products table...")
dim_products = (df_gold
    .select(
        col("product_card_id").cast("int").alias("product_card_id"),
        col("product_name"),
        col("product_price").cast("decimal(10,2)").alias("product_price"),
        col("category_name"),
        col("department_name")
    )
    .dropDuplicates(["product_card_id"])
)

# 3. CREATE DIM_DATE (Date Lookup Dimension Table)
print("📅 Constructing dim_date lookup table...")
order_dates = df_gold.select(to_date(col("order_date")).alias("date_key"))
shipping_dates = df_gold.select(to_date(col("shipping_date")).alias("date_key"))

dim_date = (order_dates.union(shipping_dates)
    .filter(col("date_key").isNotNull())
    .distinct()
    .select(
        col("date_key"),
        year(col("date_key")).cast("int").alias("year"),
        month(col("date_key")).cast("int").alias("month"),
        date_format(col("date_key"), "MMMM").alias("month_name"),
        quarter(col("date_key")).cast("int").alias("quarter"),
        dayofweek(col("date_key")).cast("int").alias("day_of_week"),
        date_format(col("date_key"), "EEEE").alias("day_name"),
        when(dayofweek(col("date_key")).isin(1, 7), lit(True)).otherwise(lit(False)).alias("is_weekend")
    )
)

# 4. CREATE FACT_SALES (Core metrics & actual/predicted target keys)
print("📈 Constructing fact_sales table...")
fact_sales = df_gold.select(
    col("order_item_id").cast("int").alias("order_item_id"),
    col("order_id").cast("int").alias("order_id"),
    col("customer_id").cast("int").alias("customer_id"),
    col("product_card_id").cast("int").alias("product_card_id"),
    to_date(col("order_date")).alias("order_date_key"),
    to_date(col("shipping_date")).alias("shipping_date_key"),
    col("type"),
    col("shipping_mode"),
    col("order_status"),
    col("order_city"),
    col("order_state"),
    col("order_country"),
    col("market"),
    col("sales").cast("decimal(10,2)").alias("sales"),
    col("order_item_quantity").cast("int").alias("order_item_quantity"),
    col("order_item_discount").cast("decimal(10,2)").alias("order_item_discount"),
    col("benefit_per_order").cast("decimal(10,2)").alias("benefit_per_order"),
    col("days_for_shipping_real").cast("int").alias("days_for_shipping_real"),
    col("days_for_shipment_scheduled").cast("int").alias("days_for_shipment_scheduled"),
    col("late_delivery_risk").cast("int").alias("late_delivery_risk"),
    col("predicted_late_delivery_risk").cast("int").alias("predicted_late_delivery_risk"),
    col("predicted_late_delivery_probability").cast("float").alias("predicted_late_delivery_probability")
)

print("✅ Star Schema splits completed successfully!")


🔄 Converting Pandas DataFrame back to Spark DataFrame...
👥 Constructing dim_customers table...
📦 Constructing dim_products table...
📅 Constructing dim_date lookup table...
📈 Constructing fact_sales table...
✅ Star Schema splits completed successfully!


## 💾 Step 6: Save Gold Delta Tables & Run Optimization
Creates the target Gold catalog schema database if it does not exist, writes the four fact/dimension tables in Delta format, and Z-Orders the fact table date keys to accelerate Tableau queries.


In [0]:
# Build schema and write paths
if target_catalog and target_catalog.lower() != "hive_metastore":
    spark.sql(f"CREATE DATABASE IF NOT EXISTS `{target_catalog}`.`{target_schema}`")
    db_prefix = f"`{target_catalog}`.`{target_schema}`"
else:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS `{target_schema}`")
    db_prefix = f"`{target_schema}`"

tables_dict = {
    "dim_customers": dim_customers,
    "dim_products": dim_products,
    "dim_date": dim_date,
    "fact_sales": fact_sales
}

# Save tables to Delta format
for table_name, df_table in tables_dict.items():
    full_path = f"{db_prefix}.`{table_name}`"
    print(f"💾 Saving Delta Table to: {full_path}")
    (df_table.write
        .format("delta")
        .mode(write_mode)
        .option("mergeSchema", "true")
        .saveAsTable(full_path)
    )
    
# Optimize storage layouts
print("⚡ Running storage optimization (Compaction & Indexes)...")
spark.sql(f"OPTIMIZE {db_prefix}.`dim_customers`")
spark.sql(f"OPTIMIZE {db_prefix}.`dim_products`")
spark.sql(f"OPTIMIZE {db_prefix}.`dim_date`")

# Z-Order fact_sales on date keys to speed up Tableau dashboard time-series charts
spark.sql(f"OPTIMIZE {db_prefix}.`fact_sales` ZORDER BY (order_date_key, shipping_date_key)")
print("⚡ Delta optimization completed successfully!")


💾 Saving Delta Table to: `workspace`.`dataco_gold`.`dim_customers`
💾 Saving Delta Table to: `workspace`.`dataco_gold`.`dim_products`
💾 Saving Delta Table to: `workspace`.`dataco_gold`.`dim_date`
💾 Saving Delta Table to: `workspace`.`dataco_gold`.`fact_sales`
⚡ Running storage optimization (Compaction & Indexes)...
⚡ Delta optimization completed successfully!


## 🔎 Step 7: Quality Checks & Assertions
Performs automated integration assertions to guarantee database row-count parity, null prediction check, and referential key integrity.


In [0]:
print("🔎 Running integration assertions...")

# 1. Row count match check
silver_rows = df_silver.count()
fact_rows = spark.read.table(f"{db_prefix}.`fact_sales`").count()
print(f"📊 Silver rows: {silver_rows:,} | Gold fact rows: {fact_rows:,}")
assert silver_rows == fact_rows, "❌ Row count mismatch between Silver and Gold fact table!"
print("✅ Row count match validation passed.")

# 2. Prediction null check
null_predictions = spark.read.table(f"{db_prefix}.`fact_sales`").filter(col("predicted_late_delivery_risk").isNull()).count()
print(f"🔎 Null predictions count: {null_predictions}")
assert null_predictions == 0, "❌ Missing predictions found in Gold fact table!"
print("✅ Null prediction check validation passed.")

# 3. Customer Key integrity check
missing_cust_keys = fact_sales.join(dim_customers, "customer_id", "leftanti").count()
print(f"🔎 Fact items missing customer dimensions: {missing_cust_keys}")
assert missing_cust_keys == 0, "❌ Key integrity violation: Customer IDs exist in fact table but not in dim_customers!"
print("✅ Customer key referential integrity check passed.")

# 4. Product Key integrity check
missing_prod_keys = fact_sales.join(dim_products, "product_card_id", "leftanti").count()
print(f"🔎 Fact items missing product dimensions: {missing_prod_keys}")
assert missing_prod_keys == 0, "❌ Key integrity violation: Product IDs exist in fact table but not in dim_products!"
print("✅ Product key referential integrity check passed.")

print("🏆 ALL QUALITY CHECKS PASSED successfully! Gold layer database is healthy.")


🔎 Running integration assertions...
📊 Silver rows: 180,519 | Gold fact rows: 180,519
✅ Row count match validation passed.
🔎 Null predictions count: 0
✅ Null prediction check validation passed.
🔎 Fact items missing customer dimensions: 0
✅ Customer key referential integrity check passed.
🔎 Fact items missing product dimensions: 0
✅ Product key referential integrity check passed.
🏆 ALL QUALITY CHECKS PASSED successfully! Gold layer database is healthy.
